# 02 Data Cleaning

This notebook cleans the raw e-commerce dataset using simple Python and pandas.

The goal is to create cleaned CSV files that can be used for validation, SQL analysis, and Power BI dashboarding.

In [ ]:
import pandas as pd
import os

## Load Raw Datasets

In [ ]:
customers = pd.read_csv("../data/raw/customers.csv")
products = pd.read_csv("../data/raw/products.csv")
orders = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")
support_tickets = pd.read_csv("../data/raw/support_tickets.csv")

## Create Copies for Cleaning

I am creating copies so that the original raw data is not changed.

In [ ]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
deliveries_clean = deliveries.copy()
support_tickets_clean = support_tickets.copy()

## Remove Duplicate Rows

In [ ]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()
deliveries_clean = deliveries_clean.drop_duplicates()
support_tickets_clean = support_tickets_clean.drop_duplicates()

In [ ]:
print("Customers duplicates:", customers_clean.duplicated().sum())
print("Products duplicates:", products_clean.duplicated().sum())
print("Orders duplicates:", orders_clean.duplicated().sum())
print("Order Items duplicates:", order_items_clean.duplicated().sum())
print("Deliveries duplicates:", deliveries_clean.duplicated().sum())
print("Support Tickets duplicates:", support_tickets_clean.duplicated().sum())

## Clean Customers Data

In [ ]:
customers_clean["customer_name"] = customers_clean["customer_name"].astype(str).str.strip().str.title()
customers_clean["gender"] = customers_clean["gender"].fillna("").astype(str).str.strip().str.title()
customers_clean["city"] = customers_clean["city"].astype(str).str.strip().str.title()
customers_clean["state"] = customers_clean["state"].astype(str).str.strip().str.title()
customers_clean["region"] = customers_clean["region"].astype(str).str.strip().str.title()
customers_clean["membership_type"] = customers_clean["membership_type"].astype(str).str.strip().str.title()

customers_clean["gender"] = customers_clean["gender"].replace("", "Not Available")

## Clean Products Data

In [ ]:
products_clean["product_name"] = products_clean["product_name"].astype(str).str.strip()
products_clean["category"] = products_clean["category"].astype(str).str.strip().str.title()
products_clean["sub_category"] = products_clean["sub_category"].astype(str).str.strip().str.title()
products_clean["brand"] = products_clean["brand"].astype(str).str.strip()
products_clean["seller_name"] = products_clean["seller_name"].astype(str).str.strip().str.title()

## Clean Orders Data

In [ ]:
orders_clean["order_status"] = orders_clean["order_status"].astype(str).str.strip().str.title()
orders_clean["sales_channel"] = orders_clean["sales_channel"].astype(str).str.strip().str.title()
orders_clean["payment_method"] = orders_clean["payment_method"].astype(str).str.strip().str.title()
orders_clean["city"] = orders_clean["city"].astype(str).str.strip().str.title()
orders_clean["state"] = orders_clean["state"].astype(str).str.strip().str.title()
orders_clean["region"] = orders_clean["region"].astype(str).str.strip().str.title()

## Clean Deliveries Data

In [ ]:
deliveries_clean["delivery_partner"] = deliveries_clean["delivery_partner"].astype(str).str.strip().str.title()
deliveries_clean["delivery_status"] = deliveries_clean["delivery_status"].astype(str).str.strip().str.title()

## Clean Support Tickets Data

In [ ]:
support_tickets_clean["issue_category"] = support_tickets_clean["issue_category"].astype(str).str.strip().str.title()
support_tickets_clean["priority"] = support_tickets_clean["priority"].astype(str).str.strip().str.title()
support_tickets_clean["ticket_status"] = support_tickets_clean["ticket_status"].astype(str).str.strip().str.title()
support_tickets_clean["issue_description"] = support_tickets_clean["issue_description"].astype(str).str.strip()

## Convert Date Columns

In [ ]:
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], errors="coerce")
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
deliveries_clean["expected_delivery_date"] = pd.to_datetime(deliveries_clean["expected_delivery_date"], errors="coerce")
deliveries_clean["actual_delivery_date"] = pd.to_datetime(deliveries_clean["actual_delivery_date"], errors="coerce")
support_tickets_clean["ticket_date"] = pd.to_datetime(support_tickets_clean["ticket_date"], errors="coerce")

## Convert Numeric Columns

In [ ]:
product_numeric_cols = ["mrp", "selling_price", "discount_percent", "cost_price", "product_rating", "rating_count"]

for col in product_numeric_cols:
    products_clean[col] = pd.to_numeric(products_clean[col], errors="coerce")

In [ ]:
order_item_numeric_cols = ["quantity", "unit_price", "discount_amount", "revenue", "profit"]

for col in order_item_numeric_cols:
    order_items_clean[col] = pd.to_numeric(order_items_clean[col], errors="coerce")

In [ ]:
deliveries_clean["delay_days"] = pd.to_numeric(deliveries_clean["delay_days"], errors="coerce")
deliveries_clean["delivery_rating"] = pd.to_numeric(deliveries_clean["delivery_rating"], errors="coerce")

support_tickets_clean["resolution_days"] = pd.to_numeric(support_tickets_clean["resolution_days"], errors="coerce")
support_tickets_clean["customer_satisfaction_score"] = pd.to_numeric(
    support_tickets_clean["customer_satisfaction_score"], 
    errors="coerce"
)

## Fix Negative Quantity

Quantity should not be zero or negative. For this beginner project, I replaced wrong quantity values with 1.

In [ ]:
order_items_clean.loc[order_items_clean["quantity"] <= 0, "quantity"] = 1
order_items_clean[order_items_clean["quantity"] <= 0]

## Fix Missing Return Reason

In [ ]:
order_items_clean["return_status"] = order_items_clean["return_status"].astype(str).str.strip().str.title()
order_items_clean["return_reason"] = order_items_clean["return_reason"].fillna("").astype(str).str.strip()

order_items_clean.loc[
    (order_items_clean["return_status"] == "Returned") & (order_items_clean["return_reason"] == ""),
    "return_reason"
] = "Reason Not Available"

In [ ]:
order_items_clean[
    (order_items_clean["return_status"] == "Returned") & 
    (order_items_clean["return_reason"] == "")
]

## Recalculate Revenue and Profit

In [ ]:
order_items_clean["revenue"] = (order_items_clean["quantity"] * order_items_clean["unit_price"]) - order_items_clean["discount_amount"]
order_items_clean.loc[order_items_clean["revenue"] < 0, "revenue"] = 0

In [ ]:
product_cost = products_clean[["product_id", "cost_price"]]

if "cost_price" in order_items_clean.columns:
    order_items_clean = order_items_clean.drop(columns=["cost_price"])

order_items_clean = order_items_clean.merge(product_cost, on="product_id", how="left")

order_items_clean["profit"] = order_items_clean["revenue"] - (
    order_items_clean["quantity"] * order_items_clean["cost_price"]
)

order_items_clean = order_items_clean.drop(columns=["cost_price"])

## Check Missing Delivery Ratings

Blank delivery rating is acceptable for cancelled, processing, or unrated orders.

In [ ]:
deliveries_clean["delivery_rating"].isnull().sum()

## Fix Satisfaction Score Range

In [ ]:
support_tickets_clean.loc[
    support_tickets_clean["customer_satisfaction_score"] > 5,
    "customer_satisfaction_score"
] = 5

support_tickets_clean.loc[
    support_tickets_clean["customer_satisfaction_score"] < 1,
    "customer_satisfaction_score"
] = 1

## Save Cleaned Files

In [ ]:
os.makedirs("../data/cleaned", exist_ok=True)

customers_clean.to_csv("../data/cleaned/customers_cleaned.csv", index=False)
products_clean.to_csv("../data/cleaned/products_cleaned.csv", index=False)
orders_clean.to_csv("../data/cleaned/orders_cleaned.csv", index=False)
order_items_clean.to_csv("../data/cleaned/order_items_cleaned.csv", index=False)
deliveries_clean.to_csv("../data/cleaned/deliveries_cleaned.csv", index=False)
support_tickets_clean.to_csv("../data/cleaned/support_tickets_cleaned.csv", index=False)

print("Cleaned datasets saved successfully.")

## Check Cleaned Data Shape

In [ ]:
print("Customers cleaned:", customers_clean.shape)
print("Products cleaned:", products_clean.shape)
print("Orders cleaned:", orders_clean.shape)
print("Order Items cleaned:", order_items_clean.shape)
print("Deliveries cleaned:", deliveries_clean.shape)
print("Support Tickets cleaned:", support_tickets_clean.shape)

## Cleaning Summary

In this step, I cleaned the raw e-commerce dataset using Python and pandas.

Main cleaning tasks completed:

1. Removed duplicate rows.
2. Removed extra spaces from text columns.
3. Standardized text formatting in city, state, region, seller, delivery partner, and issue category columns.
4. Converted date columns into proper date format.
5. Converted numeric columns into correct numeric format.
6. Fixed negative quantity values.
7. Filled missing return reason for returned items.
8. Recalculated revenue and profit after fixing quantity.
9. Saved cleaned CSV files into the data/cleaned folder.

The cleaned data is now ready for basic validation and analysis.